# Regularized Linear Models
Training Ridge (L2) and Lasso (L1) regressions with Cross-Validation.

In [ ]:
import sys
import os
import pandas as pd
import json

# Add src to path
sys.path.append('../src')

from load_data import load_raw_data
from preprocess import split_data_time_based, preprocess_data
from models import RidgeBaseline, LassoBaseline
from train import train_and_evaluate

In [ ]:
# 1. Load & Preprocess
print("Loading and Preprocessing...")
df = load_raw_data()
train_df, test_df = split_data_time_based(df)
target_col = 'hpi_metro_nsa'

train_df_scaled, test_df_scaled, scaler = preprocess_data(train_df.copy(), test_df.copy(), target_col=target_col)

cols_to_drop = ['year', 'quarter', 'metro_name', 'period_id', target_col]
X_train = train_df_scaled.drop(columns=cols_to_drop, errors='ignore')
y_train = train_df_scaled[target_col]
X_test = test_df_scaled.drop(columns=cols_to_drop, errors='ignore')
y_test = test_df_scaled[target_col]

feature_names = X_train.columns.tolist()

In [ ]:
# 2. Train Ridge
print("\n--- Ridge (L2) ---")
ridge = RidgeBaseline(alphas=[0.01, 0.1, 1, 10, 100])
train_and_evaluate(ridge, X_train, y_train, X_test, y_test, "RidgeCV")
print(f"Best Alpha: {ridge.alpha_}")

In [ ]:
# 3. Train Lasso
print("\n--- Lasso (L1) ---")
lasso = LassoBaseline(alphas=[0.01, 0.1, 1, 10, 100])
train_and_evaluate(lasso, X_train, y_train, X_test, y_test, "LassoCV")
print(f"Best Alpha: {lasso.alpha_}")

In [ ]:
# 4. Save Coefficients
coef_data = {
    "features": feature_names,
    "ridge_coefs": ridge.coef_.tolist(),
    "lasso_coefs": lasso.coef_.tolist(),
    "ridge_alpha": ridge.alpha_,
    "lasso_alpha": lasso.alpha_
}

os.makedirs('../reports/tables', exist_ok=True)
with open('../reports/tables/linear_coefficients.json', 'w') as f:
    json.dump(coef_data, f, indent=4)
print("\nCoefficients saved to reports/tables/linear_coefficients.json")